# Notebook 05: Motor de Indicadores Agroclimáticos y de Mercado (Capa Gold)
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Silver $\rightarrow$ Gold  
**Estándar**: WMO-No. 1090 (World Meteorological Organization) / FAO / DAMA-DMBOK 2  

---
### Objetivos del Motor de Indicadores (Gold)
1. **Índice de Precipitación Estandarizado (SPI)**: Ajuste paramétrico de distribución Gamma por Máxima Verosimilitud (MLE) con manejo de probabilidad cero de lluvia ($q$) y transformación normal estándar $\Phi^{-1}(H(x))$ en escalas 1, 3 y 6 meses.
2. **Anomalías Térmicas Climatológicas ($Z_T$)**: Z-scores estandarizados respecto al ciclo anual mensual.
3. **Dinámica de Precios e Insumos**: Variación interanual (YoY) y Términos de Intercambio (Terms of Trade - TOT) Insumo / Producto.
4. **Ingeniería Temporal Causal**: Rezagos cronológicos ($t-k$) y ventanas móviles puramente pasadas sin sesgo prospectivo (*no lookahead bias*).
5. **Persistencia Consolidada**: Generación del dataset multidimensional Gold en `data/INDICATORS/` con trazabilidad SHA-256.

In [1]:
# Celda 1: Setup e Importaciones del Motor de Indicadores
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.indicators.climate_indices import calculate_spi, calculate_thermal_anomaly
from src.indicators.market_indices import calculate_yoy_inflation, calculate_terms_of_trade, calculate_price_index
from src.indicators.temporal_features import add_lag_features, add_rolling_features
from src.indicators.indicator_store import IndicatorStore

logger = get_logger("notebooks.05_indicators")
store = IndicatorStore()
print(f"IndicatorStore inicializado. Directorio Gold: {store.indicators_dir}")

### 1. Cálculo del Índice de Precipitación Estandarizado (SPI-1, SPI-3, SPI-6)

In [2]:
# Celda 2: Modelado probabilístico Gamma del SPI
np.random.seed(42)
meses = pd.date_range(start="2021-01-01", periods=36, freq="ME")
# Simulación de precipitaciones mensuales en Colombia (clima bimodal)
lluvia_simulada = np.random.gamma(shape=2.5, scale=50.0, size=36)
lluvia_simulada[7] = 0.0  # Simulación de sequía severa

df_clima = pd.DataFrame({"fecha": meses, "precipitacion_mm": lluvia_simulada})
df_clima["spi_1"] = calculate_spi(df_clima["precipitacion_mm"], scale=1, min_periods=12)
df_clima["spi_3"] = calculate_spi(df_clima["precipitacion_mm"], scale=3, min_periods=12)
df_clima["spi_6"] = calculate_spi(df_clima["precipitacion_mm"], scale=6, min_periods=12)

print("Resumen estadístico de índices SPI calculados:")
display(df_clima[["fecha", "precipitacion_mm", "spi_1", "spi_3", "spi_6"]].dropna().head())

### 2. Anomalías Térmicas Climatológicas ($Z_T$)

In [3]:
# Celda 3: Cálculo de anomalías térmicas mensuales
temp_base = 22.0 + 3.0 * np.sin(np.linspace(0, 6 * np.pi, 36)) + np.random.normal(0, 1.2, 36)
df_clima["temperatura_media_c"] = temp_base
df_clima["anomalia_termica_z"] = calculate_thermal_anomaly(
    temp_series=df_clima["temperatura_media_c"],
    dates_series=df_clima["fecha"]
)

print("Anomalías térmicas Z-Score calculadas:")
display(df_clima[["fecha", "temperatura_media_c", "anomalia_termica_z"]].head())

### 3. Indicadores Económicos de Mercado (Inflación de Insumos y Términos de Intercambio)

In [4]:
# Celda 4: Variación interanual de insumos y Relación Insumo/Producto
precios_urea = 100000.0 * (1.0 + np.cumsum(np.random.normal(0.01, 0.03, 36)))
precios_papa = 2500.0 * (1.0 + np.cumsum(np.random.normal(0.005, 0.05, 36)))

df_mercado = pd.DataFrame({
    "fecha": meses,
    "precio_urea_bulto": precios_urea,
    "precio_papa_kg": precios_papa,
})
df_mercado["inflacion_yoy_urea"] = calculate_yoy_inflation(df_mercado["precio_urea_bulto"], lag_periods=12)
df_mercado["terminos_intercambio_tot"] = calculate_terms_of_trade(
    input_price_series=df_mercado["precio_urea_bulto"],
    product_price_series=df_mercado["precio_papa_kg"]
)

print("Indicadores de mercado (Muestra últimos 6 meses):")
display(df_mercado.tail(6))

### 4. Ingeniería de Características Temporales Causal (Sin Sesgo Prospectivo)

In [5]:
# Celda 5: Generación de rezagos (lags) y medias móviles rezagadas
df_mercado = add_lag_features(df_mercado, target_col="precio_papa_kg", lags=[1, 3, 6], order_by="fecha")
df_mercado = add_rolling_features(df_mercado, target_col="precio_papa_kg", windows=[3, 6], order_by="fecha")

print("Variables temporales añadidas exitosamente (Verificación de causalidad):")
display(df_mercado[["fecha", "precio_papa_kg", "precio_papa_kg_lag_1", "precio_papa_kg_roll_mean_3"]].tail(5))

### 5. Consolidación Multidimensional y Persistencia Gold en `data/INDICATORS/`

In [6]:
# Celda 6: Fusión e integridad criptográfica SHA-256 en capa Gold
df_gold = pd.merge(df_clima, df_mercado, on="fecha", how="inner")
df_gold["anio"] = df_gold["fecha"].dt.year
df_gold["mes"] = df_gold["fecha"].dt.month
df_gold["cod_municipio"] = "11001"  # Bogotá D.C. como nodo central de acopio

meta_gold = store.save_indicator_dataset(df_gold, dataset_name="indicadores_agroclimaticos_gold")
print("\n=== RESUMEN DE LA CAPA GOLD (INDICATORS) ===")
print(f"Archivo: {meta_gold['filename']}")
print(f"Filas generadas: {meta_gold['rows']}")
print(f"Columnas generadas: {meta_gold['columns']}")
print(f"Tamaño en disco: {meta_gold['file_size_bytes']} bytes")
print(f"Hash SHA-256: {meta_gold['sha256']}")